# Día 4 · Unidad 4 — Funciones (V, opcional/bonus): decoradores

> 🎁 **Contenido opcional / bonus.** El programa de la Unidad 4 no exige decoradores — este notebook amplía lo visto si el grupo va sobrado de tiempo. Si vais justos, saltad directamente a `07_ejercicios.ipynb`.

**Objetivos de aprendizaje**
- Reconocer que en Python las funciones son objetos de primera clase (*first-class objects*).
- Definir funciones anidadas y devolver funciones desde otras funciones.
- Escribir un decorador simple y aplicarlo con la sintaxis `@`.
- Decorar funciones que aceptan argumentos, y preservar su valor de retorno.
- Usar `functools.wraps` para que la función decorada conserve su identidad.

**Duración estimada:** 45-60 minutos.

**Requisitos previos:** `01_teoria_funciones_basicas.ipynb` y `02_teoria_args_kwargs.ipynb` de este mismo día.

## 1. Las funciones son objetos de primera clase

En Python, una función se puede pasar y usar como argumento, igual que cualquier otro objeto (`str`, `int`, `list`...). Esto es lo que hace posibles los decoradores.

In [ ]:
def saludo_manana(nombre: str) -> str:
    return f"Buenos días, {nombre}"


def saludo_tarde(nombre: str) -> str:
    return f"Buenas tardes, {nombre}"


In [ ]:
from typing import Callable

def saludar(func: Callable, nombre: str) -> str:
    """Aplica cualquier función de saludo sobre un nombre.

    Args:
        func: función de saludo, con la forma (str) -> str.
        nombre: nombre a saludar.

    Returns:
        str: el saludo generado por `func`.
    """
    return func(nombre)


saludar(saludo_manana, "Marta")


`saludo_manana` y `saludo_tarde` son funciones normales que esperan un `str`. `saludar()`, en cambio, espera **otra función** como argumento — eso sólo es posible porque las funciones son objetos como cualquier otro.

## 2. Funciones anidadas y funciones que devuelven funciones

Puedes definir una función dentro de otra, y esa función interna puede además ser el **valor de retorno**:

In [ ]:
def generador_recargo(tipo_cliente: str) -> Callable:
    """Devuelve la función de recargo adecuada según el tipo de cliente.

    Args:
        tipo_cliente: "nuevo" o "habitual".

    Returns:
        Callable: una función (float) -> float que aplica el recargo.
    """

    def recargo_cliente_nuevo(prima: float) -> float:
        return prima * 1.15

    def recargo_cliente_habitual(prima: float) -> float:
        return prima * 1.02

    if tipo_cliente == "nuevo":
        return recargo_cliente_nuevo
    return recargo_cliente_habitual


> **Nota**: se devuelve la función **sin** paréntesis — se devuelve la función en sí, no el resultado de llamarla.

In [ ]:
aplicar_recargo = generador_recargo("nuevo")
aplicar_recargo(300.0)


## 3. Decoradores simples

Un decorador es una función que recibe otra función, la "envuelve" en una función interna (`wrapper`), y devuelve esa envoltura.

In [ ]:
def decorador(func_a_decorar: Callable) -> Callable:
    def wrapper() -> None:
        func_a_decorar()
        print("(esta función ha sido decorada)")
    return wrapper


def notificar_cierre_dia() -> None:
    print("Cierre de cartera del día completado")


notificar_cierre_dia = decorador(notificar_cierre_dia)
notificar_cierre_dia()


Escribir `func = decorador(func)` funciona, pero repite el nombre y esconde que la función ha sido modificada. Python ofrece la sintaxis `@` para lo mismo, de forma explícita:

In [ ]:
def decorador(func_a_decorar: Callable) -> Callable:
    def wrapper() -> None:
        func_a_decorar()
        print("(esta función ha sido decorada)")
    return wrapper


@decorador
def notificar_cierre_dia() -> None:
    print("Cierre de cartera del día completado")


notificar_cierre_dia()


## 4. Decorar funciones con argumentos

¿Qué pasa si la función a decorar acepta argumentos?

In [ ]:
def repetir_dos_veces(func: Callable) -> Callable:
    def wrapper():
        func()
        func()
    return wrapper


@repetir_dos_veces
def notificar(nombre_cartera: str) -> None:
    print(f"Revisando cartera: {nombre_cartera}")


try:
    notificar("auto")
except TypeError as e:
    print(f"{type(e).__name__}: {e}")


`wrapper()` no acepta ningún argumento, pero le hemos pasado `"auto"`. La solución es que `wrapper` acepte `*args` y `**kwargs`, y se los reenvíe tal cual a la función original:

In [ ]:
def repetir_dos_veces(func: Callable) -> Callable:
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
        func(*args, **kwargs)
    return wrapper


@repetir_dos_veces
def notificar(nombre_cartera: str) -> None:
    print(f"Revisando cartera: {nombre_cartera}")


notificar("auto")


### Devolver el valor de la función decorada

Si `wrapper()` no hace `return`, el valor de retorno de la función original se pierde:

In [ ]:
@repetir_dos_veces
def resumen_cartera(nombre_cartera: str) -> str:
    print("Generando resumen...")
    return f"Resumen de {nombre_cartera} generado"


resultado = resumen_cartera("hogar")
print(resultado)


In [ ]:
def repetir_dos_veces(func: Callable) -> Callable:
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
        return func(*args, **kwargs)
    return wrapper


@repetir_dos_veces
def resumen_cartera(nombre_cartera: str) -> str:
    print("Generando resumen...")
    return f"Resumen de {nombre_cartera} generado"


resultado = resumen_cartera("hogar")
print(resultado)


## 5. `functools.wraps`

Un efecto colateral de decorar una función: su identidad (`__name__`, docstring...) pasa a ser la de `wrapper`, no la original.

In [ ]:
def calcular_prima_neta(frecuencia: float, severidad: float) -> float:
    """Calcula la prima pura (frecuencia x severidad)."""
    return frecuencia * severidad


print(calcular_prima_neta.__name__)


In [ ]:
@repetir_dos_veces
def calcular_prima_neta(frecuencia: float, severidad: float) -> float:
    """Calcula la prima pura (frecuencia x severidad)."""
    return frecuencia * severidad


print(calcular_prima_neta.__name__)  # ya no es "calcular_prima_neta"


`functools.wraps` corrige esto: aplicado sobre `wrapper`, copia el nombre, el docstring y otros metadatos de la función original.

In [ ]:
import functools

def repetir_dos_veces(func: Callable) -> Callable:
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
        return func(*args, **kwargs)
    return wrapper


@repetir_dos_veces
def calcular_prima_neta(frecuencia: float, severidad: float) -> float:
    """Calcula la prima pura (frecuencia x severidad)."""
    return frecuencia * severidad


print(calcular_prima_neta.__name__)
print(calcular_prima_neta.__doc__)


## 6. Dos decoradores útiles: `@timer` y `@debug`

### `@timer`: medir cuánto tarda una función

In [ ]:
import time

def timer(func: Callable) -> Callable:
    """Decorador que imprime cuánto tarda `func` en ejecutarse."""

    @functools.wraps(func)
    def wrapper_timer(*args, **kwargs):
        inicio = time.perf_counter()
        valor = func(*args, **kwargs)
        fin = time.perf_counter()
        print(f"{func.__name__!r} tardó {fin - inicio:.4f} s")
        return valor
    return wrapper_timer


@timer
def calcular_provision_cartera(numero_polizas: int) -> float:
    """Simula un cálculo de provisión sobre una cartera grande."""
    return sum(i * 0.001 for i in range(numero_polizas))


calcular_provision_cartera(200_000)


### `@debug`: registrar con qué argumentos se llama a una función y qué devuelve

In [ ]:
def debug(func: Callable) -> Callable:
    """Decorador que imprime la llamada (argumentos) y el valor devuelto."""

    @functools.wraps(func)
    def wrapper_debug(*args, **kwargs):
        args_repr = [repr(a) for a in args]
        kwargs_repr = [f"{k}={v!r}" for k, v in kwargs.items()]
        firma = ", ".join(args_repr + kwargs_repr)
        print(f"Llamando a {func.__name__}({firma})")
        valor = func(*args, **kwargs)
        print(f"{func.__name__!r} devuelve {valor!r}")
        return valor
    return wrapper_debug


@debug
def calcular_prima_total(numero_polizas: int, prima_unitaria: float) -> float:
    return numero_polizas * prima_unitaria


calcular_prima_total(50, 320.0)


**Para ti:** escribe un decorador `@validar_positivo` que, antes de ejecutar la función decorada, compruebe que todos los argumentos posicionales son mayores que cero (usa `*args`), y lance un `ValueError` si alguno no lo es. Pruébalo sobre `calcular_prima_total`.

## Resumen

- Las funciones son objetos de primera clase en Python: se pueden pasar como argumento, asignar a variables y devolver desde otra función.
- Una función definida dentro de otra sólo existe en el namespace de la función que la contiene.
- Un decorador es una función que recibe una función, la envuelve en un `wrapper` y devuelve ese `wrapper`; la sintaxis `@decorador` es azúcar sintáctico para `func = decorador(func)`.
- Para decorar funciones con argumentos, `wrapper` debe aceptar `*args`/`**kwargs` y reenviarlos a la función original.
- Si `wrapper` no hace `return`, se pierde el valor de retorno de la función decorada.
- `functools.wraps` conserva el nombre y el docstring originales de la función decorada — sin él, `__name__` pasa a ser `"wrapper"`.

**Siguiente:** `06_teoria_functools_bonus.ipynb` (también opcional/bonus), o `07_ejercicios.ipynb` si el grupo va justo de tiempo.